In [7]:
import os
from dotenv import load_dotenv
from langchain_google_genai import ChatGoogleGenerativeAI

load_dotenv()

model = ChatGoogleGenerativeAI(
    model="gemini-2.5-flash",
    google_api_key=os.getenv("GOOGLE_API_KEY")
)

response = model.invoke("What day is today?")

print(response.content)

Today is **Wednesday, May 22, 2024**.


In [9]:
from langchain.tools import tool

@tool
def get_weather(city : str) -> str:
    """ Gets the weather for a city """
    return f"The weather in {city} is sunny."

model_with_tools = model.bind_tools([get_weather])

LangChain's tool binding basically tells the model:

"You are allowed to use this tool."

It does not mean:

"You should execute this tool automatically."

The model sees the question and thinks:

"I need weather information. I have a weather tool available. I'll request that tool."

Because .invoke() only runs the model once. It does not automatically execute the tools that the model requests.

In [10]:
response = model_with_tools.invoke("Whats weather in ahmedabad?")
print(response)

for tool_call in response.tool_calls:
    print(f"Tool : {tool_call['name']}")
    print(f"Args : {tool_call['args']}")

content='' additional_kwargs={'function_call': {'name': 'get_weather', 'arguments': '{"city": "ahmedabad"}'}, '__gemini_function_call_thought_signatures__': {'ada555a4-33b1-4f1a-ae46-b6f0e76b22ab': 'CpgCARFNMg+z17XW8KO4wMIgCsYlcSYvjE9H+9HeXel2Xlv8sFvE1N6QCqYU79hgGE7PwQbtdGCAYs23AtlfaMxs2PgK/tG7RNH/Iwto7knQgR2zghdEf6yNhO51GjxZGAG7w+eBP2JFTx0o3FYxLRiXQJ8TRVGShOxIBHBXcveJwvfgzob0MLI6CvEWtc2L+WiF4yBIZvbBhY2cfugcYOniYRxQRSP4twRWR6qZu88nOfuuMokJRExvEmUCQhRJkHHEz6MQwfRTrhahdprIUDddhJpLA2XVkjwycaOrAh+Alx9MWCV5YaIpOYl2N3CeUKKiCUVeBjwjaDx/n8A93Q7dhajaH/c7xooX2a2abtvW1AakCDldxFBPyw=='}} response_metadata={'finish_reason': 'STOP', 'model_name': 'gemini-2.5-flash', 'safety_ratings': [], 'model_provider': 'google_genai'} id='lc_run--01a0047f-bfdf-7142-96c5-d4c0efeb15d5-0' tool_calls=[{'name': 'get_weather', 'args': {'city': 'ahmedabad'}, 'id': 'ada555a4-33b1-4f1a-ae46-b6f0e76b22ab', 'type': 'tool_call'}] invalid_tool_calls=[] usage_metadata={'input_tokens': 46, 'output_tokens': 82, 'total_tokens': 1

In [13]:
messages = [{"role" : "user", "content" : "Whats the weather in ahmedabad?"}]
ai_msg = model_with_tools.invoke(messages)
messages.append(ai_msg)

for tool_call in ai_msg.tool_calls:
    tool_result = get_weather.invoke(tool_call)
    messages.append(tool_result)

final_response = model_with_tools.invoke(messages)
print(final_response.content)    

The weather in Ahmedabad is sunny.
